In [2]:
data=[
    ("i am a student","నేను విద్యార్థిని"),
    ("how are you","మీరు ఎలా ఉన్నారు"),
    ("i love machine learning","నాకు మెషిన్ లెర్నింగ్ ఇష్టం"),
    ("good morning","శుభోదయం"),
    ("thank you","ధన్యవాదాలు"),
    ("see you later","తర్వాత కలుద్దాం"),
    ("what is your name","మీ పేరు ఏమిటి"),
    ("where are you going","మీరు ఎక్కడికి వెళుతున్నారు"),
    ("i like coffee","నాకు కాఫీ ఇష్టం"),
    ("welcome","స్వాగతం")
]

In [3]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization,Embedding,Dense,LayerNormalization,MultiHeadAttention
from tensorflow.keras import Model

In [4]:
# Separate Input and Output Sentences
english_sentences=[x[0] for x in data]
# Add Start and End Tokens
telugu_sentences=["start "+x[1]+" end" for x in data]

In [5]:
#Tokenization
vocab_size=1000 #Keep a amximum of 1000 unique words in vocabulary
sequence_length=20 #Maximum lenght of each sentence
#Tokenize English Sentences
source_vectorization=TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)
#Tokenize French Sentences
target_vectorization=TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)
#This is where learning happens.(Adapt the tokenizer to the data)
source_vectorization.adapt(english_sentences)
target_vectorization.adapt(telugu_sentences)

In [6]:
#Convert Text into Numbers
encoder_inputs=source_vectorization(english_sentences)
target_tokens=target_vectorization(telugu_sentences)

In [7]:
#Prepare Decoder Input and Output
decoder_inputs=target_tokens[:,:-1]
decoder_outputs=target_tokens[:,1:]

In [8]:
#Positional Encoding
class PositionalEncoding(tf.keras.layers.Layer):
    #Constructor
    def __init__(self,sequence_length,vocab_size,embed_dim):
        #Call the parent constructor
        super().__init__()
        #Token Embedding
        self.token_embedding=Embedding(vocab_size,embed_dim)
        #Positional Embedding
        self.position_embedding=Embedding(sequence_length,embed_dim)
        #Store Sequence Length for later use
        self.sequence_length=sequence_length
    #Call the method to compute the input sequence
    def call(self,inputs):
        #Get the Length of the Input Sequence
        length=tf.shape(inputs)[-1]
        #Create the Position Numbers
        positions=tf.range(start=0,limit=length,delta=1)
        #Convert Words to Embeddings
        embedded_tokens=self.token_embedding(inputs)
        #Convert Positions to Embeddings
        embedded_positions=self.position_embedding(positions)
        #Add Both EmbeddingsTogether
        return embedded_tokens+embedded_positions

In [9]:
#Encoder Block
#Create a Custom Encoder Layer
class TransformerEncoder(tf.keras.layers.Layer):
    #Constructor
    #embed_dim:128,dense_dim:512,num_heads:4
    #num_heads: Head1=>Grammar, Head2=>Context, Head3=>Relationships, Head4=>Meanings
    def __init__(self,embed_dim,dense_dim,num_heads):
        super().__init__()
        self.attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        #Feed Forward Network
        #Attention mixes Information
        #FFN learns complex features
        # FFN learns complex features.
        # I love AI(Input)=>Attention=>Ai is related to Love=>FFN=>(AI
        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim,
                  activation="relu"
            ),
            Dense(embed_dim)
        ])
        #Layer Normalization
        self.layernorm1=LayerNormalization()
        self.layernorm2=LayerNormalization()
    def call(self,inputs):
        #Self Attention
        attention_output=self.attention(inputs,inputs)
        #Residual Connection+LayerNorm
        proj_input=self.layernorm1(inputs+attention_output)
        #Feed Forward Network
        proj_output=self.dense_proj(proj_input)
        #Second Residual Connection+LayerNorm
        return self.layernorm2(proj_input+proj_output)


In [10]:
#Decoder Block
class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self,embed_dim,dense_dim,num_heads):
        super().__init__()
        self.self_attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        self.cross_attention=MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
        self.ffn=tf.keras.Sequential([Dense(dense_dim,activation="relu"),Dense(embed_dim)])
        #Layer Normalization
        self.layernorm1=LayerNormalization()
        self.layernorm2=LayerNormalization()
        self.layernorm3=LayerNormalization()
    def call(self,inputs,encoder_outputs):
        #Masked Self Attention
        attention_output=query=self.self_attention(query=inputs,value=inputs,key=inputs,use_causal_mask=True)
        out1=self.layernorm1(inputs+attention_output)
        #Cross Attention
        attention_output2=self.cross_attention(out1,encoder_outputs)
        out2=self.layernorm2(out1+attention_output2)
        ffn_output=self.ffn(out2)
        return self.layernorm3(out2+ffn_output)

In [18]:
#Build Complete Transformer
embed_dim=128
dense_dim=256
num_heads=4
#Encoder input Layer
encoder_input=tf.keras.Input(shape=(None,),dtype="int64")
x=PositionalEncoding(sequence_length,vocab_size,embed_dim)(encoder_input)
#Encoder Block
encoder_output=TransformerEncoder(embed_dim,dense_dim,num_heads)(x)
#Decoder Block
decoder_input=tf.keras.Input(shape=(None,),dtype="int64")
x=PositionalEncoding(sequence_length,vocab_size,embed_dim)(decoder_input)
x=TransformerDecoder(embed_dim,dense_dim,num_heads)(x,encoder_output)
decoder_output=Dense(vocab_size,activation="softmax")(x)
#Create Model
transformer=Model([encoder_input,decoder_input],decoder_output)

In [19]:
#Compile and Train
transformer.compile(optimizer='adam')
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
transformer.fit(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    batch_size=2,
    epochs=200
)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.6579 - loss: 4.1099  
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8263 - loss: 2.0601
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8263 - loss: 1.4937
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8263 - loss: 1.1367
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8526 - loss: 0.8678
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8737 - loss: 0.7192
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8632 - loss: 0.6215
Epoch 8/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.8789 - loss: 0.5368
Epoch 9/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8789 - loss: 0.4532
Epoch 10/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9211 - loss: 0.3907
Epoch 11/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8947 - loss: 0.3374
Epoch 12/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9368 -

In [25]:
#Prediction
test_sentence=["good morning"]
encoder_input_test=source_vectorization(test_sentence)
decoded_sentence="start "
for i in range(15):
    tokenized_target=target_vectorization([decoded_sentence])
    predictions=transformer.predict([encoder_input_test,tokenized_target],verbose=0)
    sampled_token_index=np.argmax(predictions[0,i,:])
    vocab=target_vectorization.get_vocabulary()
    sampled_token=vocab[sampled_token_index]
    decoded_sentence+=" "+sampled_token
    if sampled_token=="end":
        break
print("English:",test_sentence[0])
print("Telugu:",decoded_sentence.replace("start","").replace("end",""))

English: good morning
Telugu:   శుభోదయం 
